In [1]:
import json
import numpy as np
import pandas as pd

In [2]:
# ==============================================================================
# 1. CARGAR DATOS REALES DE TRÁFICO Y PARÁMETROS PREVIOS
# ==============================================================================
df_original = pd.read_csv("../datos/original_ordenado.csv")
df_original['Timestamp'] = pd.to_datetime(df_original['Timestamp'])

with open("../datos/resultados_hmm.json", "r", encoding="utf-8") as f:
    datos_hmm = json.load(f)

# Probabilidad óptima de activar Inspección Profunda (D2) desde Teoría de Juegos
prob_inspeccion_nash = 0.2717  # 27.17% obtenido en el Equilibrio de Nash

In [3]:
# 2. ESTIMACIÓN DE LA TASA DE LLEGADA (LAMBDA) DESDE EL DATASET
# ==============================================================================
# Calculamos la ventana total de captura en segundos para estimar el caudal medio
tiempo_total_seg = (df_original['Timestamp'].max() - df_original['Timestamp'].min()).total_seconds()
total_flujos = len(df_original)

# Tasa global de llegada de flujos a la red (flujos / segundo)
# Si el dataset tiene pausas muy largas entre días, también calculamos la tasa activa por hora/minuto
lambda_global = total_flujos / tiempo_total_seg

# Para modelar un escenario de tráfico sostenido (hora pico), agrupamos por minuto
flujos_por_minuto = df_original.set_index('Timestamp').resample('1min').size()
flujos_activos = flujos_por_minuto[flujos_por_minuto > 0]
lambda_pico = flujos_activos.mean() / 60.0  # Flujos por segundo en minutos con actividad

# Tasa efectiva de llegada al motor de Inspección Profunda (IPS) según Nash
lambda_ips = lambda_pico * prob_inspeccion_nash

print("--- Parámetros de Tráfico Extraídos del Dataset ---")
print(f"Total de flujos analizados        : {total_flujos:,}")
print(f"Tasa de llegada en actividad (λ)  : {lambda_pico:.4f} flujos/seg")
print(f"Proporción a Inspección (Nash)    : {prob_inspeccion_nash*100:.2f}%")
print(f"Tasa efectiva al motor IPS (λ_ef) : {lambda_ips:.4f} flujos/seg")

--- Parámetros de Tráfico Extraídos del Dataset ---
Total de flujos analizados        : 554,479
Tasa de llegada en actividad (λ)  : 16.8637 flujos/seg
Proporción a Inspección (Nash)    : 27.17%
Tasa efectiva al motor IPS (λ_ef) : 4.5819 flujos/seg


In [4]:
# 3. MODELO DE COLAS M/M/1 (Motor de Inspección Profunda de Paquetes - IPS)
# ==============================================================================
def analizar_cola_mm1(lam, mu):
    """Calcula las métricas de desempeño de un sistema de colas M/M/1."""
    if lam >= mu:
        return {"Estable": False, "rho": lam / mu}
    
    rho = lam / mu                         # Factor de utilización del servidor
    L = rho / (1 - rho)                    # Número promedio de flujos en el sistema
    Lq = (rho ** 2) / (1 - rho)            # Longitud promedio de la cola (esperando)
    W = 1 / (mu - lam)                     # Tiempo promedio total en el sistema (seg)
    Wq = lam / (mu * (mu - lam))           # Tiempo promedio de espera en cola (seg)
    P0 = 1 - rho                           # Probabilidad de que el sistema esté ocioso
    
    return {
        "Estable": True,
        "Utilización (ρ)": rho,
        "Prob. Ocioso (P0)": P0,
        "Flujos en Sistema (L)": L,
        "Flujos en Cola (Lq)": Lq,
        "Tiempo en Sistema W (seg)": W,
        "Espera en Cola Wq (seg)": Wq
    }

# Supongamos una capacidad base de servicio del IPS (mu) ligeramente superior a lambda_ips
# Por ejemplo, si el IPS procesa 1.5 veces la demanda promedio de inspección:
mu_base = lambda_ips * 1.5

metricas_base = analizar_cola_mm1(lambda_ips, mu_base)

print("\n--- Resultados del Sistema de Colas M/M/1 (Capacidad Base μ) ---")
print(f"Tasa de servicio configurada (μ)  : {mu_base:.4f} flujos/seg")
for k, v in metricas_base.items():
    if isinstance(v, float):
        print(f"  {k:28s}: {v:.4f}")    


--- Resultados del Sistema de Colas M/M/1 (Capacidad Base μ) ---
Tasa de servicio configurada (μ)  : 6.8728 flujos/seg
  Utilización (ρ)             : 0.6667
  Prob. Ocioso (P0)           : 0.3333
  Flujos en Sistema (L)       : 2.0000
  Flujos en Cola (Lq)         : 1.3333
  Tiempo en Sistema W (seg)   : 0.4365
  Espera en Cola Wq (seg)     : 0.2910


In [7]:
# 4. ANÁLISIS DE SENSIBILIDAD: IMPACTO DE LA POLÍTICA DEFENSIVA EN LA LATENCIA
# ==============================================================================
# Comparamos qué pasaría si el Defensor aplicara distintos porcentajes de inspección
politicas = [
    ("Solo Monitoreo (0% DPI)", 0.0001),
    ("Equilibrio de Nash (27.17% DPI)", prob_inspeccion_nash),
    ("Defensa Agresiva (38.00% DPI)", 0.38),
    ("Inspección Masiva (50.00% DPI)", 0.50)
]

resultados_sensibilidad = []
for nombre, p_dpi in politicas:
    lam_escenario = lambda_pico * p_dpi
    res = analizar_cola_mm1(lam_escenario, mu_base)
    if res["Estable"]:
        resultados_sensibilidad.append({
            "Política Defensiva": nombre,
            "λ_efectiva (flujos/s)": round(lam_escenario, 4),
            "Utilización ρ (%)": round(res["Utilización (ρ)"] * 100, 2),
            "Flujos en Cola (Lq)": round(res["Flujos en Cola (Lq)"], 2),
            "Latencia Total W (seg)": round(res["Tiempo en Sistema W (seg)"], 2)
        })
    else:
        resultados_sensibilidad.append({
            "Política Defensiva": nombre,
            "λ_efectiva (flujos/s)": round(lam_escenario, 4),
            "Utilización ρ (%)": round(res["rho"] * 100, 2),
            "Flujos en Cola (Lq)": "SATURADO (∞)",
            "Latencia Total W (seg)": "COLAPSO"
        })

df_sensibilidad = pd.DataFrame(resultados_sensibilidad)
print("\n--- Análisis de Sensibilidad: Saturación de Red según Política de Defensa ---")
print(df_sensibilidad.to_string(index=False))


--- Análisis de Sensibilidad: Saturación de Red según Política de Defensa ---
             Política Defensiva  λ_efectiva (flujos/s)  Utilización ρ (%) Flujos en Cola (Lq) Latencia Total W (seg)
        Solo Monitoreo (0% DPI)                 0.0017               0.02                 0.0                   0.15
Equilibrio de Nash (27.17% DPI)                 4.5819              66.67                1.33                   0.44
  Defensa Agresiva (38.00% DPI)                 6.4082              93.24               12.86                   2.15
 Inspección Masiva (50.00% DPI)                 8.4319             122.68        SATURADO (∞)                COLAPSO
